In [ ]:
import numpy as np
import sklearn.model_selection
import sklearn.metrics
import keras

First, we build the hyperparameter-optimised model and the evaluation metrics.

In [ ]:
def buildModel(input_size = 10):
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape = (input_size,)))
    model.add(keras.layers.Dense(25, activation = 'leaky_relu', kernel_regularizer = 'l1'))
    model.add(keras.layers.Dense(25, activation = 'leaky_relu', kernel_regularizer = 'l1'))
    model.add(keras.layers.Dense(25, activation = 'leaky_relu', kernel_regularizer = 'l1'))
    model.add(keras.layers.Dense(1, activation = 'sigmoid'))
    adam = keras.optimizers.Adam(learning_rate = 0.001)
    model.compile(optimizer = adam, loss = 'binary_crossentropy')
    return model

def calculateMetrics(model, datasets, verbose = False):
    for dataset in datasets:
        data_raw = np.genfromtxt(f'../data/{dataset}/snv-parse-{dataset}-knn.txt', skip_header = 1, dtype = 'str')
        x = data_raw[:, :-1].astype('float')
        y = data_raw[:, -1].astype('int')
        predictions = [1 if pred >= 0.5 else 0 for pred in model.predict(x, verbose = 0)]
        precision = sklearn.metrics.precision_score(y, predictions)
        recall = sklearn.metrics.recall_score(y, predictions)
        f1_score = sklearn.metrics.f1_score(y, predictions)
        if verbose:
            print(f'{dataset} metrics: precision - {precision}, recall - {recall}, f1 score - {f1_score}')
            print('====================')
    return [precision, recall, f1_score] # use the last dataset as the defining metric

Next, we run 30 instances of the hyperparameter-optimised model for each dataset to get the best model (WARNING: this takes a long time, run at your own risk)

In [ ]:
num_trials = 10
input_size = 10
epochs = 10
real_size = 39455 # number of rows of data from real datasets
files = ['full', 'full_no_syn5']
datasets = ['syn1', 'syn2', 'syn3', 'syn4', 'syn5', 'real1', 'real2_part1']
best_results = []
for f in files:
    # get training and testing data
    output = []
    for subset in ['train', 'test']:
        data_raw = np.genfromtxt(f'../data/full_no_syn5/snv-parse-full_no_syn5-knn-{subset}.txt', dtype = 'str')
        output.append(data_raw[:, :-1].astype('float'))
        output.append(data_raw[:, -1].astype('int'))
    x_train, y_train, x_test, y_test = output
    # set different sample weight bias based on how many synthetic datasets are used
    sample_weight_bias = 5 if f == 'full' else 4
    sample_weight = np.ones(x_train.shape[0])
    sample_weight[-real_size:] = sample_weight_bias
    results = []
    # run 30 iterations of model training and evaluation
    for i in range(30):
        model = buildModel(input_size = input_size)
        callback = keras.callbacks.EarlyStopping(patience = 3, verbose = 0, restore_best_weights = True)
        model.fit(x_train, y_train, epochs = epochs, verbose = 0, callbacks = [callback], validation_data = (x_test, y_test), sample_weight = sample_weight)
        results.append([model] + calculateMetrics(model, datasets))
        print(f'{f} run {i + 1} completed')
    results.sort(key = lambda x: x[3], reverse = True)
    best_results.append(results)
# get best results
for f in files:
    results = best_results.pop(0)
    # results[0][0].save(f'./models/{f}-best_model.keras')
    print(f'{f} overall metrics: precision - {results[0][1]}, recall - {results[0][2]}, f1 score - {results[0][3]}')
    print(f'average f1: {sum([x[3] for x in results])/30}')
    print('========================================')

To see model performance of the current best model, run the code below.

In [ ]:
datasets = ['syn1', 'syn2', 'syn3', 'syn4', 'syn5', 'real1', 'real2_part1']
calculateMetrics(keras.models.load_model(f'../models/full_no_syn5-best_model.keras'), datasets = datasets, verbose = True)